In [0]:
def test_duplicate_txn(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, lag, unix_timestamp

    data = [
        ("C1", "Amazon", 1000, "2024-01-01 10:00:00"),
        ("C1", "Amazon", 1000, "2024-01-01 10:03:00")
    ]

    columns = ["customer_id", "merchant", "amount", "txn_time"]
    df = spark.createDataFrame(data, columns)

    df = df.withColumn("txn_ts", unix_timestamp("txn_time"))

    window_spec = Window.partitionBy("customer_id", "merchant", "amount") \
                        .orderBy("txn_ts")

    df = df.withColumn("prev_txn_ts", lag("txn_ts").over(window_spec)) \
           .withColumn("time_diff_sec", col("txn_ts") - col("prev_txn_ts"))

    result_df = df.filter(col("time_diff_sec") <= 300)

    try:
        assert result_df.count() == 1
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")

test_duplicate_txn(spark)